In [ ]:
from datasets import load_dataset
import torch 
from transformers import AutoTokenizer 
from Model.model import GPTModel
from trainer import Trainer
from torch.utils.data import DataLoader, TensorDataset,Dataset

In [2]:
device = torch.device("cuda")
tokenizer = AutoTokenizer.from_pretrained(r"C:\Users\itsN3Fi\Desktop\Model\Tokenizer")

In [3]:

device = torch.device("cuda")
vocab_size = len(tokenizer) 
D_model = 1024
n_heads= 16 
n_layers = 12
n_groups=8
dropout=0.1
model = GPTModel(vocab_size,D_model,n_heads,n_layers,n_groups,None,dropout).to(device)

print(f"{sum(p.numel() for p in model.parameters())}")
print(f"Model Parameters: {sum(p.numel() for p in model.parameters()):,}")

221550080
Model Parameters: 221,550,080


In [ ]:
dataset = load_dataset("roneneldan/TinyStories", split="train")

dataset = dataset.select(range(int(0.01* len(dataset))))

Tokenize Dataset

In [5]:


max_len = 512


class PackedDataset(Dataset):
    def __init__(self, tokenized_texts, block_size=1024):
        """
        tokenized_texts: list[list[int]]
        """
        self.block_size = block_size


        all_tokens = []
        for t in tokenized_texts:
            all_tokens.extend(t)

        self.data = torch.tensor(all_tokens, dtype=torch.long)

        self.num_chunks = len(self.data) // block_size

    def __len__(self):
        return self.num_chunks

    def __getitem__(self, idx):
        start = idx * self.block_size
        end = start + self.block_size

        chunk = self.data[start:end]

        x = chunk[:-1]
        y = chunk[1:]

        return x, y
    
def tokenize_fn(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=max_len,
        padding=False  
    )

tokenized = dataset.map(tokenize_fn, batched=True, remove_columns=["text"])


token_list = tokenized["input_ids"]

dataset = PackedDataset(token_list, block_size=512)

dataloader = DataLoader(
    dataset,
    batch_size=4,
    shuffle=True,
    drop_last=True
)

Train model

In [ ]:
trainer = Trainer(model=model,dataloader=dataloader,checkpoint_step=1000,device='cuda')

In [ ]:
trainer.Train()

Example OutPut

In [10]:
input_ids = tokenizer.encode("hello",return_tensors='pt').to(device)
pred = model.generate_manual(input_ids)
tokenizer.decode(pred[0])

'hello how you are the largest compelling’s Global Global television.\nS.\nGlying, these to be seen on the plant in the island of the key lab from your place. They can find the design of your own application, but if weigh the ability to progress.\nEvenven much more broad and you have a result of place to the public’s understanding of the topic.\nNow, these, you may be able to allow their stairment and enjoyable.'